In [0]:
-- Create the ZeroBus target table used by lakeLoom transcript/event ingestion.
--
-- This task is intentionally idempotent so the platform bootstrap job can be
-- re-run safely. It creates the expected bronze table contract before the
-- final validation step.
--
-- Parameters supplied via notebook job task base_parameters:
--   catalog_use, schema_use, secret_scope_name, client_id_dbs_key, spn_application_id
-- These are accessed via $param_name syntax (widget substitution).

SELECT
  current_timestamp() AS ddl_started_at,
  '$catalog_use' AS target_catalog,
  '$schema_use' AS target_schema,
  CONCAT('$catalog_use', '.', '$schema_use', '.transcript_events_raw') AS target_table_name;

In [0]:
-- CREATE TABLE ... does not support IDENTIFIER() with $widget references.
-- Use EXECUTE IMMEDIATE so the runtime substitutes the widget values.

EXECUTE IMMEDIATE "CREATE TABLE IF NOT EXISTS $catalog_use.$schema_use.transcript_events_raw
(
  event_id STRING NOT NULL COMMENT 'Producer-generated idempotency key or event identifier',
  session_id STRING COMMENT 'Application session identifier associated with the transcript/event payload',
  project_id STRING COMMENT 'Application project identifier associated with the transcript/event payload',
  user_id STRING COMMENT 'End-user identifier carried in the event payload for attribution beyond the shared SPN actor',
  device_id STRING COMMENT 'Paired-device identifier or label when available',
  event_type STRING NOT NULL COMMENT 'Logical event type such as partial_transcript, final_transcript, audio_uploaded, or client_status',
  event_time TIMESTAMP COMMENT 'Event timestamp supplied by the producer when available',
  ingested_at TIMESTAMP NOT NULL COMMENT 'Server-side ingest timestamp written by the producer or connector',
  transcript_text STRING COMMENT 'Transcript text for transcript-bearing events',
  transcript_language STRING COMMENT 'Language code associated with the transcript text when available',
  source_platform STRING COMMENT 'Origin platform for the event, such as ios',
  workspace_id STRING COMMENT 'Workspace identifier embedded in the event payload when available',
  headers VARIANT COMMENT 'Captured request metadata and non-sensitive headers as semi-structured JSON',
  body VARIANT COMMENT 'Full raw ZeroBus event payload stored as VARIANT for flexible bronze retention',
  CONSTRAINT transcript_events_raw_pk PRIMARY KEY (event_id)
)
USING DELTA
COMMENT 'Bronze ZeroBus target for lakeLoom transcript and session event ingestion from paired iOS devices'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.enableRowTracking' = 'true',
  'delta.enableVariantShredding' = 'true',
  'quality' = 'bronze',
  'pipeline' = 'lakeloom_platform_bootstrap',
  'description' = 'ZeroBus streaming target table for transcript and pairing-related session events'
);"

In [0]:
EXECUTE IMMEDIATE "SHOW CREATE TABLE $catalog_use.$schema_use.transcript_events_raw";

In [0]:
SELECT
  'transcript_events_raw_ready' AS check_name,
  'created_or_confirmed' AS status,
  CONCAT('$catalog_use', '.', '$schema_use', '.transcript_events_raw') AS target_table_name;

In [0]:
-- Capture spn_application_id injected by upstream task for use in GRANT statements.
DECLARE OR REPLACE VARIABLE spn_application_id STRING DEFAULT '$spn_application_id';
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT '$catalog_use';
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT '$schema_use';

SELECT spn_application_id, catalog_use, schema_use;

In [0]:
DECLARE OR REPLACE VARIABLE use_catalog_grnt_stmnt STRING DEFAULT
  CONCAT('GRANT USE CATALOG ON CATALOG ', catalog_use, ' TO `', spn_application_id, '`;');

SELECT use_catalog_grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE use_catalog_grnt_stmnt;

In [0]:
DECLARE OR REPLACE VARIABLE use_schema_grnt_stmnt STRING DEFAULT
  CONCAT('GRANT USE SCHEMA ON SCHEMA ', catalog_use, '.', schema_use, ' TO `', spn_application_id, '`;');

SELECT use_schema_grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE use_schema_grnt_stmnt;

In [0]:
DECLARE OR REPLACE VARIABLE tbl_grnt_stmnt STRING DEFAULT
  CONCAT('GRANT MODIFY, SELECT ON TABLE ', catalog_use, '.', schema_use, '.transcript_events_raw TO `', spn_application_id, '`;');

SELECT tbl_grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE tbl_grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE CONCAT('SHOW GRANTS ON TABLE ', catalog_use, '.', schema_use, '.transcript_events_raw');